In [1]:
import os
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

from tqdm import tqdm

from PIL import Image

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jonathanoheix/face-expression-recognition-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\princ\.cache\kagglehub\datasets\jonathanoheix\face-expression-recognition-dataset\versions\1


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [4]:
import os
from PIL import Image
from torch.utils.data import Dataset

class ExpressionDataset(Dataset):
    def __init__(self, dataset_path, transform=None):
        self.transform = transform
        
        # Get class folders
        self.classes = [(i, f) for i, f in enumerate(os.listdir(dataset_path)) 
                        if os.path.isdir(os.path.join(dataset_path, f))]

        self._img_with_labels = []
        image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

        for label, cls_name in self.classes:
            cls_folder = os.path.join(dataset_path, cls_name)
            images = [f for f in os.listdir(cls_folder)
                      if f.lower().endswith(image_extensions) and os.path.isfile(os.path.join(cls_folder, f))]

            for img_name in images:
                img_path = os.path.join(cls_folder, img_name)
                # Only store path and label, do NOT load image yet
                self._img_with_labels.append((img_path, label))

    def __len__(self):
        return len(self._img_with_labels)

    def __getitem__(self, idx):
        img_path, label = self._img_with_labels[idx]
        img = Image.open(img_path).convert("L")  # Load image only when needed
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

In [5]:
train_dataset = ExpressionDataset(os.path.join(path, "images", "train"), transform)
val_dataset = ExpressionDataset(os.path.join(path, "images", "validation"), transform)

In [6]:
train_dataset[28820]

(tensor([[[ 0.6706,  0.7098,  0.6784,  ...,  0.9843,  0.9765,  0.9922],
          [ 0.6392,  0.6706,  0.7020,  ...,  0.9843,  0.9765,  0.9765],
          [ 0.6784,  0.6784,  0.7176,  ...,  0.9765,  0.9843,  0.9765],
          ...,
          [ 0.6392,  0.5216,  0.3882,  ...,  0.9686,  1.0000,  1.0000],
          [ 0.6471,  0.4902,  0.5294,  ...,  0.1608,  0.4275,  0.7412],
          [ 0.5373,  0.4824,  0.6314,  ..., -0.2392, -0.2784, -0.3490]]]),
 tensor(6))

In [7]:
class ExpressionClassifier(nn.Module):

    # -----------------------
    # Shared Block: Conv → BN → ReLU
    # -----------------------
    def conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    # -----------------------
    # Branch 1: Standard CNN
    # -----------------------
    class DeepStructureCNN(nn.Module):
        def __init__(self, input_channel, hidden_channel):
            super().__init__()
            self.block1 = nn.Sequential(
                nn.Conv2d(input_channel, hidden_channel, kernel_size=3, padding=1),
                nn.BatchNorm2d(hidden_channel),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2)
            )
            self.block2 = nn.Sequential(
                nn.Conv2d(hidden_channel, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2)
            )
            # Global avg pool to remove huge flatten
            self.gap = nn.AdaptiveAvgPool2d((1, 1))

        def forward(self, x):
            x = self.block1(x)
            x = self.block2(x)
            x = self.gap(x)
            return x.view(x.size(0), -1)   # -> [B, 64]

    # -----------------------
    # Branch 2: AntiAliasing Path
    # -----------------------
    class AntiAliasing(nn.Module):
        def __init__(self, input_channel, hidden_channel):
            super().__init__()
            self.block1 = nn.Sequential(
                nn.Conv2d(input_channel, hidden_channel, kernel_size=3, padding=1),
                nn.BatchNorm2d(hidden_channel),
                nn.ReLU(inplace=True),
                nn.AvgPool2d(2, 2)
            )
            self.block2 = nn.Sequential(
                nn.Conv2d(hidden_channel, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.AvgPool2d(2, 2)
            )
            self.gap = nn.AdaptiveAvgPool2d((1, 1))

        def forward(self, x):
            x = self.block1(x)
            x = self.block2(x)
            x = self.gap(x)
            return x.view(x.size(0), -1)   # -> [B, 64]

    # -----------------------
    # Combined Model
    # -----------------------
    def __init__(self, input_channel, hidden_channel, output_channel):
        super().__init__()
        self.branch1 = self.DeepStructureCNN(input_channel, hidden_channel)
        self.branch2 = self.AntiAliasing(input_channel, hidden_channel)

        # combined feature size is ALWAYS 64 + 64 = 128
        self.fc1 = nn.Linear(128, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, output_channel)

    def forward(self, x):
        x1 = self.branch1(x)     # -> [B, 64]
        x2 = self.branch2(x)     # -> [B, 64]
        x = torch.cat([x1, x2], dim=1)  # -> [B, 128]

        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [9]:
img, label = next(iter(train_loader))
img.shape

torch.Size([32, 1, 48, 48])

In [10]:
input_channel = img.shape[1]
hidden_channel = 32
output_channel = len(train_dataset.classes)

# model, loss, optimizer
model = ExpressionClassifier(input_channel, hidden_channel, output_channel)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, 
                epochs=50, patience=5):
    model.to(device)
    writer = SummaryWriter(log_dir="runs/expression-classifier")

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(1, epochs + 1):
        # -----------------------
        # Training phase
        # -----------------------
        model.train()
        running_loss = 0.0
        running_corrects = 0
        total = 0
        
        for images, labels in tqdm(train_loader, desc=f"Train Epoch {epoch}"):
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            total += labels.size(0)
        
        epoch_loss = running_loss / total
        epoch_acc = running_corrects.double() / total
        
        # -----------------------
        # Validation phase
        # -----------------------
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Val Epoch {epoch}"):
                images = images.to(device)
                labels = labels.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_total += labels.size(0)
        
        val_epoch_loss = val_loss / val_total
        val_epoch_acc = val_corrects.double() / val_total
        
        print(f"Epoch {epoch}/{epochs} | "
              f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | "
              f"Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:.4f}")
        
        writer.add_scalar("Loss/Train", epoch_loss, epoch)
        writer.add_scalar("Loss/Val", val_epoch_loss, epoch)
        writer.add_scalar("Accuracy/Train", epoch_acc, epoch)
        writer.add_scalar("Accuracy/Val", val_epoch_acc, epoch)
        
        if val_epoch_loss < best_val_loss:
            best_val_loss = val_epoch_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    # Load best model weights
    model.load_state_dict(best_model_wts)
    writer.close()
    return model


In [12]:
expression_classifier = train_model(model, train_loader, val_loader, criterion, optimizer, device, 500, 10)

Val Epoch 1: 100%|███████████████████████████████████████████████████████████████████| 442/442 [00:44<00:00, 10.03it/s]


Epoch 1/500 | Train Loss: 1.7801 | Train Acc: 0.2598 | Val Loss: 1.7501 | Val Acc: 0.2830


Val Epoch 2: 100%|███████████████████████████████████████████████████████████████████| 442/442 [00:31<00:00, 14.07it/s]


Epoch 2/500 | Train Loss: 1.7506 | Train Acc: 0.2802 | Val Loss: 1.7213 | Val Acc: 0.3181


Val Epoch 3: 100%|███████████████████████████████████████████████████████████████████| 442/442 [00:32<00:00, 13.71it/s]


...

Epoch 35/500 | Train Loss: 1.3944 | Train Acc: 0.4683 | Val Loss: 1.4375 | Val Acc: 0.4454


Val Epoch 36: 100%|██████████████████████████████████████████████████████████████████| 442/442 [00:20<00:00, 21.10it/s]


Epoch 36/500 | Train Loss: 1.3918 | Train Acc: 0.4657 | Val Loss: 1.4704 | Val Acc: 0.4387


Train Epoch 37:  42%|██████████████████████████▌                                     | 374/901 [01:48<02:32,  3.46it/s]


RuntimeError: [enforce fail at alloc_cpu.cpp:121] data. DefaultCPUAllocator: not enough memory: you tried to allocate 9437184 bytes.

In [ ]:
torch.save(expression_classifier.state_dict(), "expression_model.pth")

In [ ]:
train_dataset.classes